# DenseNet-121 Multi-GPU Training Job 제출
- 인스턴스: ml.g5.12xlarge (A10G x4)
- 배치: 128 (GPU당 32)
- Spot 학습, 예상 ~5시간

In [ ]:
import boto3

sm = boto3.client('sagemaker', region_name='ap-northeast-2')

sm.create_training_job(
    TrainingJobName='densenet121-full-pa-v6-multigpu',
    RoleArn='arn:aws:iam::666803869796:role/SKKU_SageMaker_Role',
    AlgorithmSpecification={
        'TrainingImage': '763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker',
        'TrainingInputMode': 'File'
    },
    HyperParameters={
        'sagemaker_program': 'train_multigpu.py',
        'sagemaker_submit_directory': 's3://pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an/code/densenet_multigpu_sourcedir.tar.gz',
        'sagemaker_region': 'ap-northeast-2',
        'batch-size': '128',
        'stage1-epochs': '5',
        'stage2-epochs': '25',
        'image-bucket': 'say1-pre-project-5',
        'work-bucket': 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
    },
    ResourceConfig={
        'InstanceType': 'ml.g5.12xlarge',
        'InstanceCount': 1,
        'VolumeSizeInGB': 150
    },
    InputDataConfig=[{
        'ChannelName': 'csv',
        'DataSource': {
            'S3DataSource': {
                'S3DataType': 'S3Prefix',
                'S3Uri': 's3://pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an/mimic-cxr-csv/',
                'S3DataDistributionType': 'FullyReplicated'
            }
        }
    }],
    CheckpointConfig={
        'S3Uri': 's3://pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an/checkpoints/densenet121-full-pa-v6-multigpu/'
    },
    OutputDataConfig={
        'S3OutputPath': 's3://pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an/output/'
    },
    EnableManagedSpotTraining=True,
    StoppingCondition={
        'MaxRuntimeInSeconds': 21600,
        'MaxWaitTimeInSeconds': 172800
    },
    Tags=[{
        'Key': 'name',
        'Value': 'say2-preproject-6team-hyunwoo'
    }]
)

print('v6 제출 완료!')

## 상태 확인

In [ ]:
resp = sm.describe_training_job(TrainingJobName='densenet121-full-pa-v6-multigpu')
print(f"상태: {resp['TrainingJobStatus']}")
print(f"세부: {resp['SecondaryStatus']}")
print(f"소요: {resp.get('TrainingTimeInSeconds', 0) // 60}분")